# Driver Snapshot Analysis

## Objective

This notebook analyzes daily driver snapshots to evaluate:

- Snapshot consistency
- Driver status changes
- Change Data Capture (CDC) opportunities
- Slowly Changing Dimension (SCD) behavior
- Incremental loading strategy

The findings will be used to design an efficient CDC pipeline and dimension management strategy.

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [2]:
DATA_PATH = Path("../data/raw")

snapshot_day1 = pd.read_csv(
    DATA_PATH / "drivers_snapshot_2026-05-08.csv"
)

snapshot_day2 = pd.read_csv(
    DATA_PATH / "drivers_snapshot_2026-05-09.csv"
)

In [3]:
print("Day 1 shape:", snapshot_day1.shape)
print("Day 2 shape:", snapshot_day2.shape)

Day 1 shape: (9, 7)
Day 2 shape: (9, 7)


In [4]:
snapshot_day1.columns.tolist()

['driver_id',
 'driver_name',
 'city_id',
 'vehicle_type',
 'rating',
 'driver_status',
 'snapshot_date']

In [5]:
snapshot_day1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   driver_id      9 non-null      int64  
 1   driver_name    9 non-null      object 
 2   city_id        9 non-null      int64  
 3   vehicle_type   9 non-null      object 
 4   rating         9 non-null      float64
 5   driver_status  9 non-null      object 
 6   snapshot_date  9 non-null      object 
dtypes: float64(1), int64(2), object(4)
memory usage: 632.0+ bytes


In [6]:
snapshot_day1.head()

,driver_id,driver_name,city_id,vehicle_type,rating,driver_status,snapshot_date
0,3001,Ahmet Y.,34,comfort,4.82,active,2026-05-08
1,3002,Mehmet K.,34,economy,4.55,active,2026-05-08
2,3003,Ayse D.,34,comfort,4.90,active,2026-05-08
3,3004,Fatma S.,34,premium,4.71,active,2026-05-08
4,3005,Can T.,34,comfort,4.63,active,2026-05-08


In [7]:
snapshot_day1.columns.equals(snapshot_day2.columns)

True

In [8]:
day1_drivers = set(snapshot_day1["driver_id"])
day2_drivers = set(snapshot_day2["driver_id"])

new_drivers = day2_drivers - day1_drivers
removed_drivers = day1_drivers - day2_drivers
common_drivers = day1_drivers & day2_drivers

print(f"New drivers: {len(new_drivers)}")
print(f"Removed drivers: {len(removed_drivers)}")
print(f"Common drivers: {len(common_drivers)}")

print("New driver IDs:", new_drivers)
print("Removed driver IDs:", removed_drivers)

New drivers: 0
Removed drivers: 0
Common drivers: 9
New driver IDs: set()
Removed driver IDs: set()


In [9]:
comparison = snapshot_day1.merge(
    snapshot_day2,
    on="driver_id",
    how="inner",
    suffixes=("_day1", "_day2")
)

comparison.head()

,driver_id,driver_name_day1,city_id_day1,vehicle_type_day1,rating_day1,driver_status_day1,snapshot_date_day1,driver_name_day2,city_id_day2,vehicle_type_day2,rating_day2,driver_status_day2,snapshot_date_day2
0,3001,Ahmet Y.,34,comfort,4.82,active,2026-05-08,Ahmet Y.,34,comfort,4.82,active,2026-05-09
1,3002,Mehmet K.,34,economy,4.55,active,2026-05-08,Mehmet K.,34,comfort,4.55,active,2026-05-09
2,3003,Ayse D.,34,comfort,4.90,active,2026-05-08,Ayse D.,34,comfort,4.90,active,2026-05-09
3,3004,Fatma S.,34,premium,4.71,active,2026-05-08,Fatma S.,34,premium,4.71,active,2026-05-09
4,3005,Can T.,34,comfort,4.63,active,2026-05-08,Can T.,34,comfort,4.63,active,2026-05-09


In [10]:
tracked_columns = [
    "driver_name",
    "city_id",
    "vehicle_type",
    "rating",
    "driver_status"
]

In [11]:
change_mask = False

for column in tracked_columns:
    change_mask = change_mask | (
        comparison[f"{column}_day1"]
        != comparison[f"{column}_day2"]
    )

changed_drivers = comparison[change_mask]

changed_drivers

,driver_id,driver_name_day1,city_id_day1,vehicle_type_day1,rating_day1,driver_status_day1,snapshot_date_day1,driver_name_day2,city_id_day2,vehicle_type_day2,rating_day2,driver_status_day2,snapshot_date_day2
1,3002,Mehmet K.,34,economy,4.55,active,2026-05-08,Mehmet K.,34,comfort,4.55,active,2026-05-09
5,3006,Deniz A.,35,comfort,4.44,active,2026-05-08,Deniz A.,34,comfort,4.44,active,2026-05-09
7,3009,Burak N.,34,comfort,4.30,active,2026-05-08,Burak N.,34,comfort,4.38,active,2026-05-09
8,3011,Zeynep O.,34,economy,4.58,active,2026-05-08,Zeynep O.,34,economy,4.58,suspended,2026-05-09


In [12]:
changes = []

for _, row in changed_drivers.iterrows():
    for column in tracked_columns:
        old_value = row[f"{column}_day1"]
        new_value = row[f"{column}_day2"]

        if old_value != new_value:
            changes.append({
                "driver_id": row["driver_id"],
                "attribute": column,
                "old_value": old_value,
                "new_value": new_value
            })

changes_df = pd.DataFrame(changes)

changes_df

,driver_id,attribute,old_value,new_value
0,3002,vehicle_type,economy,comfort
1,3006,city_id,35,34
2,3009,rating,4.3,4.38
3,3011,driver_status,active,suspended


In [17]:
changes_df["attribute"].value_counts(dropna=False)

vehicle_type     1
city_id          1
rating           1
driver_status    1
Name: attribute, dtype: int64

In [14]:
print(f"Common drivers: {len(comparison)}")
print(f"Changed drivers: {changed_drivers['driver_id'].nunique()}")
print(f"Total attribute changes: {len(changes_df)}")

Common drivers: 9
Changed drivers: 4
Total attribute changes: 4


In [15]:
print(f"New drivers: {len(new_drivers)}")
print(f"Removed drivers: {len(removed_drivers)}")
print(f"Common drivers: {len(common_drivers)}")

print("New driver IDs:", new_drivers)
print("Removed driver IDs:", removed_drivers)

New drivers: 0
Removed drivers: 0
Common drivers: 9
New driver IDs: set()
Removed driver IDs: set()


# Findings

## Snapshot Comparison

The source provides consecutive daily full-load driver snapshots rather than CDC events.

The two driver snapshots contain the same nine drivers. No new or removed drivers were identified between the snapshot dates.

Four drivers contain changes in business attributes:

- Driver 3002 changed `vehicle_type` from economy to comfort.
- Driver 3006 changed `city_id` from 35 to 34.
- Driver 3009 changed `rating` from 4.30 to 4.38.
- Driver 3011 changed `driver_status` from active to suspended.

## SCD Strategy

A hybrid dimension management strategy is recommended.

### SCD Type 2

Historical versions should be maintained for:

- `vehicle_type`
- `city_id`
- `driver_status`

These attributes may affect historical trip analysis, operational reporting and audit requirements.

### SCD Type 1 or Separate History Dataset

The `rating` attribute is a frequently changing metric and should not automatically create a new driver dimension version for every update.

The latest rating can be maintained using SCD Type 1. If historical rating analysis is required, rating changes should be stored in a dedicated history dataset.

## Snapshot-Based Change Detection

Because the source provides daily full-load snapshots rather than CDC events, changes must be derived by comparing consecutive snapshots.

`driver_id` should be used as the business key.

A hash of the tracked business attributes can be calculated for each driver. Only new or changed hash values should proceed to the dimension merge process.

This approach avoids unnecessary updates and enables incremental dimension processing despite receiving full source snapshots.